# 11 · Autodifusão da água (MSD) com correção de tamanho finito

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** O modelo de água da simulação difunde na velocidade certa? (complementa a validação estrutural do notebook 07: um modelo pode acertar a g(r) e errar a dinâmica)

Relação de Einstein: $\mathrm{MSD}(\tau) = \langle |\mathbf r(t+\tau) - \mathbf r(t)|^2\rangle \to 6D\tau$.

* **Correção de tamanho finito** (Yeh & Hummer, *J. Phys. Chem. B* **2004**, 108, 15873): sob
  condições periódicas, $D_{PBC}$ é menor que o do sistema infinito,
  $D_\infty = D_{PBC} + \dfrac{\xi\,k_B T}{6\pi\eta L}$, com $\xi = 2{,}837297$ (caixa cúbica) e
  $\eta$ a viscosidade **do modelo** de água (não a experimental). Em caixa não cúbica, $L = V^{1/3}$ é
  aproximação.
* **Referência:** $D = 2{,}299 \times 10^{-9}$ m² s⁻¹ a 25 °C (Mills, *J. Phys. Chem.* **1973**, 77,
  685). Para outras temperaturas, use o valor experimental correspondente.
* **Com proteína na caixa**, as águas da camada de hidratação difundem mais devagar e o $D$ médio
  sai abaixo do da água pura. A comparação limpa com o experimento é numa **caixa de água pura**
  do mesmo modelo (aponte `PASTA_BASE`/`ARQUIVO_TRAJ` para ela); no sistema, o número é didático.

**Desdobramento.** O `md_300ns_mol.xtc` (`-pbc mol -center`) recoloca as moléculas na caixa a cada frame.
As posições são desdobradas aqui somando os deslocamentos de imagem mínima entre frames
consecutivos (válido enquanto o deslocamento entre frames é muito menor que meia caixa — use
`DT_PS` ≤ 10 ps) e removendo o deslocamento médio das águas (o `-center` e a deriva do centro de
massa, como o `-rmcomm` do `gmx msd`). Com uma trajetória `-pbc nojump`, o desdobramento não muda
nada.

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mdtraj as md
import MDAnalysis as mda
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "MDTraj": md.__version__,
    "MDAnalysis": mda.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
# ── Onde estão as simulações ────────────────────────────────────────────────
PASTA_BASE = Path("..")             # pasta que contém rep1/, rep2/, ...   <- AJUSTE
SISTEMA = "xilanase_1XNB"           # nome da pasta de resultados
REPLICAS = [1, 2, 3, 4, 5]          # uma réplica só também funciona: [1]
PREFIXO_MD = "md_300ns"             # -deffnm da produção


def pasta_replica(n):
    """Pasta da réplica n."""
    return PASTA_BASE / f"rep{n}"

ARQUIVO_TOP = f"{PREFIXO_MD}.gro"            # sistema inteiro
ARQUIVO_TRAJ = f"{PREFIXO_MD}_mol.xtc"        # sistema inteiro, -pbc mol -center
ARQUIVO_TPR = f"{PREFIXO_MD}.tpr"            # elementos pelo nº atômico (MDAnalysis >= 2.11); None = pelo nome
ARQUIVOS = {"topologia": ARQUIVO_TOP, "trajetória": ARQUIVO_TRAJ, **({"tpr": ARQUIVO_TPR} if ARQUIVO_TPR else {})}

# ── Amostragem ──────────────────────────────────────────────────────────────
DT_PS = 10.0                    # intervalo entre frames (o desdobramento exige frames próximos)
DESCARTE_NS = 0.0
JANELA_NS = 10.0                # trecho da produção usado (o MSD converge rápido; limita a RAM)
NIVEL_CONFIANCA = 0.95

# ── Análise ─────────────────────────────────────────────────────────────────
OXIGENIO_AGUA = "water and name O"
TAU_MAX_PS = 1000.0             # maior intervalo de tempo do MSD
FAIXA_AJUSTE_PS = (20.0, 200.0) # região linear (fora do regime balístico e da cauda ruidosa)
N_BLOCOS = 5                    # D por bloco da janela -> erro de cada réplica
TEMPERATURA_K = 298.15
D_EXP_M2_S = 2.299e-9           # Mills 1973, 25 °C
XI_YEH_HUMMER = 2.837297        # caixa cúbica
# Viscosidade de cisalhamento DO MODELO (mPa s) a 298 K — González & Abascal, J. Chem. Phys. 2010, 132, 096101:
VISCOSIDADES_MODELO = {"TIP3P": 0.321, "SPC/E": 0.729, "TIP4P/2005": 0.855, "OPC": 0.8269}
MODELO_AGUA = "OPC"             # <- o seu modelo; se não estiver na tabela, defina VISCOSIDADE_MPA_S
VISCOSIDADE_MPA_S = VISCOSIDADES_MODELO.get(MODELO_AGUA)       # OPC: η da SUA caixa de água pura; None = sem correção
D_MODELO_M2_S = 2.393e-09            # D∞ do mesmo modelo na caixa de água pura (referência do próprio modelo); None = não usa
K_BOLTZMANN = 1.380649e-23

# ── Saída ───────────────────────────────────────────────────────────────────
DIR_SAIDA = Path("resultados") / SISTEMA / "11_difusao_agua"
PREFIXO = "difusao"
TITULO = "11 · Autodifusão da água"

In [ ]:
linhas = []
for n in REPLICAS:
    for papel, nome in ARQUIVOS.items():
        caminho = pasta_replica(n) / nome
        linhas.append({"replica": f"rep{n}", "arquivo": papel, "caminho": str(caminho),
                       "existe": caminho.exists(),
                       "tamanho_MB": round(caminho.stat().st_size / 1024 ** 2, 1) if caminho.exists() else np.nan})
df_arquivos = pd.DataFrame(linhas)
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima). Ajuste RAIZ/SISTEMA/REPLICAS.")

In [ ]:
MAPA_RESNAMES = {"HIE": "HIS", "HID": "HIS", "HIP": "HIS", "HSD": "HIS", "HSE": "HIS",
                 "HSP": "HIS", "CYX": "CYS", "CYM": "CYS", "LYN": "LYS", "ASH": "ASP",
                 "GLH": "GLU"}


def tpr_de(n):
    """Caminho do .tpr da réplica n (None se ARQUIVO_TPR = None)."""
    return pasta_replica(n) / ARQUIVO_TPR if ARQUIVO_TPR else None


def elementos_do_tpr(tpr, top, arquivo):
    """Elemento de cada átomo da topologia, pelo número atômico gravado no .tpr.

    O .gro pode ser o sistema inteiro ou um subconjunto na MESMA ordem (o _prot.gro).
    O casamento é resíduo a resíduo, pela sequência de nomes de átomos: o MDAnalysis
    renumera os resíduos do .tpr a partir de 1, então o nº do resíduo (Leu4...) não serve.
    """
    u = mda.Universe(str(tpr))
    if not hasattr(u.atoms, "elements"):
        raise ValueError("o .tpr não trouxe números atômicos")
    if str(arquivo).endswith(".gro"):            # nomes crus (o MDTraj renomeia OW -> O etc.)
        linhas = Path(arquivo).read_text().splitlines()[2:2 + top.n_atoms]
        nomes = [l[10:15].strip() for l in linhas]
    else:
        nomes = [a.name for a in top.atoms]
    cortes = np.flatnonzero(np.diff(u.atoms.resindices)) + 1
    residuos_tpr = list(zip(np.split(u.atoms.names, cortes), np.split(u.atoms.elements, cortes)))
    elementos, j = [], 0
    for residuo in top.residues:
        alvo = [nomes[a.index] for a in residuo.atoms]
        while j < len(residuos_tpr) and list(residuos_tpr[j][0]) != alvo:
            j += 1
        if j == len(residuos_tpr):
            raise ValueError(f"{residuo} não casa com o .tpr (o .gro não é um subconjunto ordenado dele)")
        elementos += list(residuos_tpr[j][1])
        j += 1
    return elementos


def ler_topologia(arquivo, tpr=None):
    """Topologia MDTraj de um .gro/.pdb, com os elementos certos.

    * Com `tpr` (MDAnalysis >= 2.11 lê o .tpr do GROMACS 2026): o elemento vem do número
      atômico gravado no .tpr — a mesma informação que o GROMACS usa.
    * Sem `tpr` (ou se a leitura falhar): o MDTraj deduz o elemento pelo nome do átomo, e o
      FE do heme viraria flúor (19 u em vez de 55,85 u); ele é corrigido e um aviso é impresso.
    * Resíduos com nome de protonação (HIE, CYM...) recebem o nome padrão, para que
      `protein` e `resname` funcionem.
    """
    top = md.load_topology(str(arquivo))
    for residuo in top.residues:
        residuo.name = MAPA_RESNAMES.get(residuo.name.upper(), residuo.name)
    if tpr is not None and Path(tpr).exists():
        try:
            elementos = elementos_do_tpr(tpr, top, arquivo)
            for atomo, simbolo in zip(top.atoms, elementos):
                atomo.element = md.element.get_by_symbol(simbolo.capitalize()) if simbolo else md.element.virtual
            return top
        except Exception as erro:
            print(f"[aviso] .tpr não usado ({erro}); elementos deduzidos pelo nome do átomo.")
    for atomo in top.atoms:
        if atomo.residue.name == "HEM" and atomo.name.upper() == "FE":
            atomo.element = md.element.iron
    return top


def selecionar(top, consulta, n=None):
    """top.select com erro explícito se vazio (ou se não tiver n átomos)."""
    idx = top.select(consulta)
    if len(idx) == 0:
        raise ValueError(f"Seleção vazia: '{consulta}' (no MDTraj o nº do resíduo é `resSeq`).")
    if n is not None and len(idx) != n:
        raise ValueError(f"'{consulta}' casou com {len(idx)} átomos (esperado {n}).")
    return idx


def intervalo_do_arquivo_ps(xtc, top):
    """Δt entre frames gravados no .xtc (lê só os 2 primeiros frames)."""
    for lote in md.iterload(str(xtc), top=top, chunk=2, atom_indices=[0]):
        return float(np.diff(lote.time)[0])


def estride(xtc, top, dt_alvo_ps):
    """Passo de leitura que leva o Δt do arquivo ao Δt de análise."""
    dt = intervalo_do_arquivo_ps(xtc, top)
    passo = max(1, int(round(dt_alvo_ps / dt)))
    if dt_alvo_ps < dt:
        print(f"[aviso] Δt pedido {dt_alvo_ps:g} ps é menor que o do arquivo ({dt:g} ps).")
    elif abs(passo * dt - dt_alvo_ps) > 1e-6 * dt_alvo_ps:
        print(f"[aviso] Δt {dt_alvo_ps:g} ps não é múltiplo de {dt:g} ps; usando {passo * dt:g} ps.")
    return passo, dt


def producao(t_ns):
    """Máscara dos frames de produção (depois de DESCARTE_NS)."""
    t_ns = np.asarray(t_ns, dtype=float)
    mascara = t_ns >= t_ns[0] + DESCARTE_NS
    if not mascara.any():
        raise ValueError("DESCARTE_NS é maior que a trajetória.")
    return mascara



linhas, dados = [], {}
for n in REPLICAS:
    rep, xtc = f"rep{n}", pasta_replica(n) / ARQUIVO_TRAJ
    top = ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    ow = selecionar(top, OXIGENIO_AGUA)
    passo, dt = estride(xtc, top, DT_PS)
    traj = md.load(str(xtc), top=top, stride=passo, atom_indices=ow)
    t_ns = np.asarray(traj.time) / 1000.0
    janela = producao(t_ns) & (t_ns <= t_ns[producao(t_ns)][0] + JANELA_NS)
    traj = traj[janela]
    dados[rep] = traj
    linhas.append({"replica": rep, "aguas": ow.size, "dt_ps": passo * dt, "frames_janela": traj.n_frames,
                   "janela_ns": f"{traj.time[0] / 1000:.1f}–{traj.time[-1] / 1000:.1f}",
                   "volume_medio_nm3": float(traj.unitcell_volumes.mean()), "memoria_MB": round(traj.xyz.nbytes / 1024 ** 2, 1)})
df_trajetorias = pd.DataFrame(linhas)
df_trajetorias

## 3 · Cálculos

Desdobramento → MSD por FFT, com todas as origens de tempo (algoritmo de Calandrini *et al.*,
*Collection SFN* **2011**, 12, 201: $\mathrm{MSD}(m) = S_1(m) - 2S_2(m)$, com $S_2$ a
autocorrelação das posições) → média sobre as águas.

In [ ]:
def desdobrar(traj):
    """Posições contínuas (F, N, 3): soma os deslocamentos de imagem mínima entre frames (caixa
    triclínica) e remove o deslocamento médio do grupo a cada passo."""
    x = traj.xyz.astype(np.float64)
    caixa = traj.unitcell_vectors.astype(np.float64)
    d = np.diff(x, axis=0)
    for f in range(d.shape[0]):
        s = d[f] @ np.linalg.inv(caixa[f + 1])          # coordenadas fracionárias
        d[f] = (s - np.round(s)) @ caixa[f + 1]
    d -= d.mean(axis=1, keepdims=True)                  # remove -center e deriva do centro de massa
    return np.concatenate([x[:1], x[:1] + np.cumsum(d, axis=0)])


def msd_fft(x, n_max):
    """MSD médio (n_max + 1,) de x (F, N, 3), todas as origens, por FFT."""
    F = x.shape[0]
    r2 = (x ** 2).sum(axis=2)                           # (F, N)
    tam = 1 << (2 * F - 1).bit_length()
    soma_s2 = np.zeros(F)
    for k in range(3):
        f = np.fft.rfft(x[:, :, k], n=tam, axis=0)
        soma_s2 += np.fft.irfft(f * np.conjugate(f), n=tam, axis=0)[:F].sum(axis=1)
    s2 = soma_s2 / (x.shape[1] * (F - np.arange(F)))
    q = 2 * r2.sum(axis=1)
    s1 = np.empty(F)
    acumulado = q.sum() / x.shape[1]
    for m in range(F):
        if m:
            acumulado -= (r2[m - 1].sum() + r2[F - m].sum()) / x.shape[1]
        s1[m] = acumulado / (F - m)
    return (s1 - 2 * s2)[:n_max + 1]


quadros, blocos = [], []
for rep, traj in dados.items():
    x = desdobrar(traj)
    dt_ps = float(np.median(np.diff(traj.time)))
    n_max = min(int(TAU_MAX_PS / dt_ps), x.shape[0] - 1)
    quadros.append(pd.DataFrame({"replica": rep, "tau_ps": np.arange(n_max + 1) * dt_ps, "msd_nm2": msd_fft(x, n_max)}))
    for b, fatia in enumerate(np.array_split(np.arange(x.shape[0]), N_BLOCOS), start=1):
        nb = min(n_max, len(fatia) - 1)
        blocos.append(pd.DataFrame({"replica": rep, "bloco": b, "tau_ps": np.arange(nb + 1) * dt_ps,
                                    "msd_nm2": msd_fft(x[fatia], nb)}))
    print(f"[{rep}] MSD até {n_max * dt_ps:.0f} ps")
df_msd = pd.concat(quadros, ignore_index=True)
df_msd_blocos = pd.concat(blocos, ignore_index=True)
df_msd.groupby("replica", sort=False)["msd_nm2"].describe()

## 4 · Estatística

$D$ = inclinação/6 do ajuste linear em `FAIXA_AJUSTE_PS` (nm² ps⁻¹ → m² s⁻¹: × 10⁻⁶). Erro de
cada réplica: dispersão de $D$ entre os blocos da janela. Entre réplicas: média ± SEM. A
correção de Yeh–Hummer usa $L = \langle V\rangle^{1/3}$ de cada réplica. Comparação com o
experimento em unidades do SEM entre réplicas.

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
def ajustar_d(tau, msd):
    faixa = (tau >= FAIXA_AJUSTE_PS[0]) & (tau <= FAIXA_AJUSTE_PS[1])
    if faixa.sum() < 3:
        raise ValueError("Poucos pontos em FAIXA_AJUSTE_PS: aumente a janela ou reduza DT_PS.")
    ajuste = stats.linregress(tau[faixa], msd[faixa])
    return ajuste.slope / 6.0 * 1e-6                   # nm²/ps -> m²/s


linhas = []
for rep, g in df_msd.groupby("replica", sort=False):
    d_pbc = ajustar_d(g["tau_ps"].to_numpy(), g["msd_nm2"].to_numpy())
    d_bl = [ajustar_d(b["tau_ps"].to_numpy(), b["msd_nm2"].to_numpy())
            for _, b in df_msd_blocos[df_msd_blocos["replica"] == rep].groupby("bloco")]
    L = float(df_trajetorias.set_index("replica").loc[rep, "volume_medio_nm3"]) ** (1 / 3) * 1e-9      # m
    correcao = (XI_YEH_HUMMER * K_BOLTZMANN * TEMPERATURA_K / (6 * np.pi * VISCOSIDADE_MPA_S * 1e-3 * L)
                if VISCOSIDADE_MPA_S else np.nan)
    linhas.append({"replica": rep, "D_pbc_m2_s": d_pbc, "sem_blocos_m2_s": np.std(d_bl, ddof=1) / np.sqrt(len(d_bl)),
                   "L_nm": L * 1e9, "correcao_yeh_hummer_m2_s": correcao, "D_infinito_m2_s": d_pbc + correcao})
df_difusao = pd.DataFrame(linhas)
obs = "D∞ (Yeh–Hummer)" if VISCOSIDADE_MPA_S else "D_PBC"
coluna_d = "D_infinito_m2_s" if VISCOSIDADE_MPA_S else "D_pbc_m2_s"
if not VISCOSIDADE_MPA_S:
    print(f"[!] Sem viscosidade para '{MODELO_AGUA}': D sem correção de tamanho finito (subestimado).")
linhas = [{"observavel": obs, "replica": r, "unidade": "m² s⁻¹", "media": v, "sem": s}
          for r, v, s in df_difusao[["replica", coluna_d, "sem_blocos_m2_s"]].itertuples(index=False)]
if len(df_difusao) > 1:
    linhas.append(entre_replicas(df_difusao[coluna_d], obs, "m² s⁻¹"))
df_stats = pd.DataFrame(linhas)
ref = referencia(df_stats, obs)
df_comparacao = pd.DataFrame([{"grandeza": obs, "simulado_m2_s": ref["media"], "sem_m2_s": ref["sem"],
                               "experimental_m2_s": D_EXP_M2_S, "razao_sim_exp": ref["media"] / D_EXP_M2_S,
                               "desvio_em_sem": (ref["media"] - D_EXP_M2_S) / ref["sem"]}])
if D_MODELO_M2_S:
    df_comparacao["modelo_bulk_m2_s"] = D_MODELO_M2_S
    df_comparacao["razao_sim_modelo"] = ref["media"] / D_MODELO_M2_S
    print(f"Razão sistema/água pura do mesmo modelo: {ref['media'] / D_MODELO_M2_S:.3f} (abaixo de 1 = água retardada pela proteína)")
if len(REPLICAS) == 1:
    print("[1 réplica] Não há linha 'entre_replicas': o erro é o SEM da própria série, "
          "corrigido pela ineficiência estatística (subestima a incerteza de observáveis lentos).")
print(f"{obs} = {formatar(ref['media'] * 1e9, ref['sem'] * 1e9)} × 10⁻⁹ m² s⁻¹ | experimental {D_EXP_M2_S * 1e9:.3f} × 10⁻⁹ "
      f"| razão {ref['media'] / D_EXP_M2_S:.2f}")
display(df_difusao)
df_comparacao

## 5 · Gráfico (matplotlib)

A: MSD por réplica com a região do ajuste sombreada e a reta de $D$ experimental (pontilhado).
B: o mesmo em log-log, com a reta de inclinação 1: o ajuste só vale onde as curvas a acompanham
(Maginn *et al.*, *LiveCoMS* **2019**, 1, 6324). C: $D$ por réplica (± EP entre blocos da
janela), a média do sistema (± EP entre réplicas), a água pura do mesmo modelo (notebook 14,
traço-ponto) e o experimental (pontilhado vermelho). A razão sistema/água pura isola o efeito da
proteína; a razão água pura/experimental, o do modelo.

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Arial"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 10                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 6.3, 3.37

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig, (ax_m, ax_l, ax_d) = plt.subplots(1, 3, figsize=(LARGURA, ALTURA), layout="constrained", gridspec_kw={"width_ratios": [1.3, 1.1, 1]})
for i, (rep, g) in enumerate(df_msd.groupby("replica", sort=False)):
    ax_m.plot(g["tau_ps"], g["msd_nm2"], color=CORES[i % len(CORES)], lw=0.9, label=rep)
tau = df_msd["tau_ps"].drop_duplicates().to_numpy()
ax_m.plot(tau, 6 * D_EXP_M2_S * 1e6 * tau, color=COR_REFERENCIA, ls=":", lw=0.9, label="6 D$_{exp}$ τ")
ax_m.axvspan(*FAIXA_AJUSTE_PS, color="#969696", alpha=0.15, lw=0)
ax_m.set(xlabel="τ (ps)", ylabel="MSD (nm$^2$)")
ax_m.legend(frameon=True, fontsize=TAMANHO_FONTE - 2)
# log-log (Maginn et al. 2019): no regime difusivo a inclinação é 1; o regime balístico fica à esquerda
for i, (rep, g) in enumerate(df_msd.groupby("replica", sort=False)):
    g = g[g["tau_ps"] > 0]
    ax_l.loglog(g["tau_ps"], g["msd_nm2"], color=CORES[i % len(CORES)], lw=0.9)
meio = np.sqrt(FAIXA_AJUSTE_PS[0] * FAIXA_AJUSTE_PS[1])
ancora_msd = float(np.interp(meio, tau, df_msd.groupby("tau_ps")["msd_nm2"].mean().reindex(tau).to_numpy()))
ax_l.loglog(tau[tau > 0], ancora_msd * tau[tau > 0] / meio, color=COR_MEDIA, ls="--", lw=0.8, label="inclinação 1")
ax_l.axvspan(*FAIXA_AJUSTE_PS, color="#969696", alpha=0.15, lw=0)
ax_l.set(xlabel="τ (ps)", ylabel="MSD (nm$^2$)")
ax_l.legend(frameon=True, fontsize=TAMANHO_FONTE - 2)
# C: D por réplica (± EP entre blocos da janela), média do sistema (± EP entre réplicas) e referências
COR_MODELO = "#58508d"
x = np.arange(len(df_difusao))
d9 = df_difusao[coluna_d].to_numpy() * 1e9
e9 = df_difusao["sem_blocos_m2_s"].to_numpy() * 1e9
for i in x:
    ax_d.errorbar(i, d9[i], yerr=e9[i], fmt="o", ms=4, color=CORES[i % len(CORES)], capsize=2, lw=0.8)
ax_d.axhline(ref["media"] * 1e9, color=COR_MEDIA, ls="--", lw=0.8,
             label=f"sistema: {formatar(ref['media'] * 1e9, ref['sem'] * 1e9)} (média ± EP)")
if D_MODELO_M2_S:
    ax_d.axhline(D_MODELO_M2_S * 1e9, color=COR_MODELO, ls="-.", lw=0.9,
                 label=f"{MODELO_AGUA} pura (nb. 14): {D_MODELO_M2_S * 1e9:.3f}")
ax_d.axhline(D_EXP_M2_S * 1e9, color=COR_REFERENCIA, ls=":", lw=0.9,
             label=f"experimental: {D_EXP_M2_S * 1e9:.3f} (Mills, 1973)")
valores = np.r_[d9, D_EXP_M2_S * 1e9, (D_MODELO_M2_S or np.nan) * 1e9]
folga = 0.15 * (np.nanmax(valores) - np.nanmin(valores))
ax_d.set(xticks=x, xticklabels=df_difusao["replica"], xlim=(-0.5, len(x) - 1.0),
         ylim=(np.nanmin(valores) - folga, np.nanmax(valores) + folga),
         ylabel=f"{'D$_∞$' if VISCOSIDADE_MPA_S else 'D$_{PBC}$'} (10$^{{-9}}$ m$^2$ s$^{{-1}}$)")
ax_d.legend(frameon=True, fontsize=TAMANHO_FONTE - 2, loc="upper center", bbox_to_anchor=(0.5, -0.12))
for eixo, letra in zip((ax_m, ax_l, ax_d), "ABC"):
    eixo.set_title(letra, loc="left", fontweight="bold")
fig_difusao = fig
plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 420                # pixels
LARGURA_PX = None              # None = largura da página
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
# legenda: só as réplicas; a reta experimental e a faixa de ajuste estão rotuladas no gráfico
fig_i = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=("linear", "log-log"))
for col in (1, 2):
    for i, (rep, g) in enumerate(df_msd.groupby("replica", sort=False)):
        g = g[g["tau_ps"] > 0] if col == 2 else g
        fig_i.add_trace(go.Scatter(x=g["tau_ps"], y=g["msd_nm2"], mode="lines", name=rep, legendgroup=rep,
                                   showlegend=(col == 1), line=dict(color=CORES[i % len(CORES)])), row=1, col=col)
    fig_i.add_vrect(x0=FAIXA_AJUSTE_PS[0], x1=FAIXA_AJUSTE_PS[1], fillcolor="#969696", opacity=0.15, line_width=0,
                    row=1, col=col)
fig_i.add_trace(go.Scatter(x=tau, y=6 * D_EXP_M2_S * 1e6 * tau, mode="lines", name="6 D_exp τ", showlegend=False,
                           line=dict(color=COR_REFERENCIA, dash="dot")), row=1, col=1)
fig_i.add_annotation(x=tau[-1], y=6 * D_EXP_M2_S * 1e6 * tau[-1], text="6 D<sub>exp</sub> τ (Mills 1973)",
                     showarrow=False, xanchor="right", yanchor="bottom", font=dict(color=COR_REFERENCIA), row=1, col=1)
fig_i.add_trace(go.Scatter(x=tau[tau > 0], y=ancora_msd * tau[tau > 0] / meio, mode="lines", name="inclinação 1",
                           showlegend=False, line=dict(color=COR_MEDIA, dash="dash")), row=1, col=2)
fig_i.add_annotation(x=np.log10(tau[-1]), y=np.log10(ancora_msd * tau[-1] / meio), text="inclinação 1", showarrow=False,
                     xanchor="right", yanchor="bottom", row=1, col=2)
fig_i.update_xaxes(title_text="τ (ps)")
fig_i.update_xaxes(type="log", row=1, col=2)
fig_i.update_yaxes(title_text="MSD (nm²)", row=1, col=1)
fig_i.update_yaxes(type="log", row=1, col=2)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, width=LARGURA_PX, legend_title_text="Réplica",
                    font=dict(family=FONTE, size=TAMANHO_FONTE_PX))
fig_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `msd` | MSD(τ) médio das águas por réplica |
| `msd_por_bloco` | MSD(τ) por bloco da janela |
| `difusao` | D_PBC, erro por blocos, L, correção de Yeh–Hummer e D∞ por réplica |
| `estatistica` | D por réplica e `entre_replicas` |
| `comparacao_experimental` | simulado × Mills (1973): razão e desvio em SEM |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"difusao_agua": fig_difusao}
INTERATIVOS = {"difusao_agua": fig_interativo}
TABELAS = {"msd": df_msd, "msd_por_bloco": df_msd_blocos, "difusao": df_difusao, "estatistica": df_stats,
          "comparacao_experimental": df_comparacao}
PARAMETROS = {"descarte_ns": DESCARTE_NS, "dt_ps": DT_PS, "janela_ns": JANELA_NS, "faixa_ajuste_ps": FAIXA_AJUSTE_PS,
              "modelo_agua": MODELO_AGUA, "viscosidade_mPa_s": VISCOSIDADE_MPA_S, "temperatura_k": TEMPERATURA_K}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados